# 01 — Get the data

**Project:** Do U.S. climate change concerns move European green-minus-brown returns?
**Author:** Rafael Machado Cerqueira — Advanced Data Analytics, HEC Lausanne, Autumn 2026

This notebook downloads every raw series used in the project and saves it, untouched, in `data/raw/`.
No cleaning happens here: that is the job of `02_clean.ipynb`.

| Series | Source | Licence / terms |
|---|---|---|
| Media Climate Change Concerns index (daily) | Sentometrics Research — https://sentometrics-research.com/download/mccc/ | Free for academic research; cite Ardia et al. (2023, *Management Science*) and footnote https://sentometrics-research.com |
| European green & brown stocks, U.S. ETFs, FX rates | Yahoo Finance via `yfinance` | Yahoo terms of use — data not redistributed, fetched by this script |
| VIX (`VIXCLS`), Brent (`DCOILBRENTEU`) | FRED, Federal Reserve Bank of St. Louis | Public |
| Fama-French 3 factors, U.S. and Europe (daily) | Kenneth French Data Library | Free for research |

Because some of these terms do not allow redistribution, `data/raw/` is listed in `.gitignore`:
the repository holds this script, not the files (Requirement 4.15).

*Code drafted with Claude (Anthropic) and reviewed by the author — declared in the paper's appendix.*

## 0. Setup

In [4]:
%pip install -r ../requirements.txt
import io
import json
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

# Paths relative to the repository root, whether the notebook runs from code/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "code" else Path.cwd()
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Sample period (the MCCC 2025 vintage ends in June 2025)
START = "2010-01-01"
END = "2025-07-01"   # exclusive in yfinance -> last day 30 June 2025

print("Saving raw files to:", RAW)


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Saving raw files to: /Users/RafaelMachadoCerqueira/Desktop/Data/climate-concerns-green-brown/data/raw


## 1. Media Climate Change Concerns index (MCCC)

2025 vintage only, over the whole sample: the methodology changed between vintages 2 and 3 (mid-2018),
so mixing vintages would create an artificial break.

In [5]:
MCCC_URL = "https://sentometrics-research.com/data/mccc/mccc-2025-update-daily.csv.gz"

r = requests.get(MCCC_URL, timeout=60)
r.raise_for_status()
mccc = pd.read_csv(io.BytesIO(r.content), compression="gzip")
mccc.to_csv(RAW / "mccc_daily.csv", index=False)

print(mccc.shape)
print(list(mccc.columns))
mccc.head()

(8217, 36)
['Date', 'Aggregate', 'cluster_Business Impact', 'cluster_Environmental Impact', 'cluster_Societal Debate', 'cluster_Research', 'Agreements/Actions', 'Agriculture Shifts', 'Airline Industry', 'Arctic Wildlife', 'Car Industry', 'Carbon Credits Market', 'Carbon Reduction Technologies', 'Carbon Tax', 'Cities', 'Climate Legislation/Regulations', 'Climate Summits', 'Controversies', 'Corporations/Investments', 'Ecosystems', 'Extreme Temperatures', 'Food Shortage/Poverty', 'Forests', 'Glaciers/Ice Sheets', 'Global Warming', 'Government Programs', 'Hurricanes/Floods', 'Legal Actions', 'Marine Wildlife', 'Political Campaign', 'Renewable Energy', 'Scientific Studies', 'Social Events', 'Tourism', 'UN/IPCC Reports', 'Water/Drought']


,Date,Aggregate,cluster_Business Impact,cluster_Environmental Impact,cluster_Societal Debate,cluster_Research,Agreements/Actions,Agriculture Shifts,Airline Industry,Arctic Wildlife,...,Hurricanes/Floods,Legal Actions,Marine Wildlife,Political Campaign,Renewable Energy,Scientific Studies,Social Events,Tourism,UN/IPCC Reports,Water/Drought
0,2003-01-01,0.298420,0.357690,0.143705,0.305688,0.042479,0.082278,0.197260,0.085954,0.059739,...,0.044658,0.289730,0.039871,0.234461,0.066717,0.038917,0.336510,0.104707,0.042475,0.064698
1,2003-01-02,0.478648,0.088762,0.626138,0.233561,0.576001,0.116337,0.152607,0.060875,0.092622,...,0.118157,0.083055,0.184832,0.064826,0.047225,0.565352,0.285850,0.255777,0.354847,0.150958
2,2003-01-03,0.370424,0.048446,0.501245,0.080927,0.457464,0.048578,0.071360,0.078565,0.038172,...,0.158486,0.014842,0.127090,0.018111,0.030351,0.592481,0.100897,0.295819,0.089824,0.138514
3,2003-01-04,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,2003-01-05,0.099583,0.098087,0.106874,0.061362,0.030494,0.018942,0.045565,0.027592,0.080238,...,0.087821,0.117991,0.075307,0.016761,0.092316,0.040483,0.035450,0.146881,0.023010,0.086962


## 2. Stock and ETF prices (yfinance)

Candidate universe — to be finalised after looking at coverage below.
Green/brown classification is by sector (renewable generation and equipment vs. oil & gas);
the justification goes in the paper.

In [6]:
GREEN_EU = {
    "VWS.CO":    "Vestas Wind Systems",
    "ORSTED.CO": "Orsted",
    "IBE.MC":    "Iberdrola",
    "EDPR.LS":   "EDP Renovaveis",
    "NDX1.DE":   "Nordex",
    "SLR.MC":    "Solaria",
}
BROWN_EU = {
    "SHEL.L":  "Shell",
    "BP.L":    "BP",
    "TTE.PA":  "TotalEnergies",
    "ENI.MI":  "Eni",
    "EQNR.OL": "Equinor",
    "REP.MC":  "Repsol",
}
US_ETF = {
    "ICLN": "iShares Global Clean Energy (green benchmark)",
    "XLE":  "Energy Select Sector SPDR (brown benchmark)",
    "SPY":  "S&P 500 (U.S. market control)",
}
FX = {
    "EURUSD=X": "EUR/USD",
    "EURGBP=X": "EUR/GBP",
    "EURDKK=X": "EUR/DKK",
    "EURNOK=X": "EUR/NOK",
}

universe = pd.DataFrame(
    [(t, n, "green_eu") for t, n in GREEN_EU.items()]
    + [(t, n, "brown_eu") for t, n in BROWN_EU.items()]
    + [(t, n, "us_etf") for t, n in US_ETF.items()]
    + [(t, n, "fx") for t, n in FX.items()],
    columns=["ticker", "name", "group"],
)
universe

,ticker,name,group
0,VWS.CO,Vestas Wind Systems,green_eu
1,ORSTED.CO,Orsted,green_eu
2,IBE.MC,Iberdrola,green_eu
3,EDPR.LS,EDP Renovaveis,green_eu
4,NDX1.DE,Nordex,green_eu
5,SLR.MC,Solaria,green_eu
6,SHEL.L,Shell,brown_eu
7,BP.L,BP,brown_eu
8,TTE.PA,TotalEnergies,brown_eu
9,ENI.MI,Eni,brown_eu


In [7]:
# Adjusted closes (dividends and splits included), one column per ticker
px = yf.download(
    universe["ticker"].tolist(),
    start=START, end=END,
    auto_adjust=True, progress=False,
)["Close"]

px.to_csv(RAW / "prices_daily.csv")
print(px.shape)

(4040, 19)


Quote currency of each ticker. Needed in `02_clean` for FX conversion —
and note that London listings (`.L`) are quoted in **pence (GBp)**, not pounds.

In [8]:
def get_currency(ticker):
    try:
        return yf.Ticker(ticker).fast_info["currency"]
    except Exception:
        return None

universe["currency"] = universe["ticker"].map(get_currency)
universe.to_csv(RAW / "universe.csv", index=False)
universe

,ticker,name,group,currency
0,VWS.CO,Vestas Wind Systems,green_eu,DKK
1,ORSTED.CO,Orsted,green_eu,DKK
2,IBE.MC,Iberdrola,green_eu,EUR
3,EDPR.LS,EDP Renovaveis,green_eu,EUR
4,NDX1.DE,Nordex,green_eu,EUR
5,SLR.MC,Solaria,green_eu,EUR
6,SHEL.L,Shell,brown_eu,GBp
7,BP.L,BP,brown_eu,GBp
8,TTE.PA,TotalEnergies,brown_eu,EUR
9,ENI.MI,Eni,brown_eu,EUR


### Coverage check

Which tickers returned nothing, and when does each series really start?
This decides the start of the sample (a late IPO shortens it for everyone).

In [9]:
coverage = pd.DataFrame({
    "first_date": px.apply(lambda s: s.first_valid_index()),
    "last_date":  px.apply(lambda s: s.last_valid_index()),
    "n_obs":      px.notna().sum(),
}).join(universe.set_index("ticker")[["name", "group"]])

missing = sorted(set(universe["ticker"]) - set(coverage.index[coverage["n_obs"] > 0]))
print("Tickers with no data:", missing or "none")
coverage.sort_values("first_date")

Tickers with no data: none


,first_date,last_date,n_obs,name,group
Ticker,,,,,
EURDKK=X,2010-01-01,2025-06-30,4036,EUR/DKK,fx
EURGBP=X,2010-01-01,2025-06-30,4036,EUR/GBP,fx
EURNOK=X,2010-01-01,2025-06-30,4036,EUR/NOK,fx
EURUSD=X,2010-01-01,2025-06-30,4034,EUR/USD,fx
BP.L,2010-01-04,2025-06-30,3911,BP,brown_eu
TTE.PA,2010-01-04,2025-06-30,3964,TotalEnergies,brown_eu
SPY,2010-01-04,2025-06-30,3896,S&P 500 (U.S. market control),us_etf
SLR.MC,2010-01-04,2025-06-30,3963,Solaria,green_eu
SHEL.L,2010-01-04,2025-06-30,3912,Shell,brown_eu


## 3. Controls from FRED: VIX and Brent

In [11]:
def get_fred(series_id):
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}"
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    df = pd.read_csv(io.StringIO(r.text), na_values=".")
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")

fred = get_fred("VIXCLS").join(get_fred("DCOILBRENTEU"), how="outer")
fred = fred.loc[START:END]
fred.to_csv(RAW / "fred_controls_daily.csv")
fred.tail()

,VIXCLS,DCOILBRENTEU
date,,
2025-06-25,16.76,68.40
2025-06-26,16.59,68.57
2025-06-27,16.32,69.37
2025-06-30,16.73,68.15
2025-07-01,16.83,67.63


## 4. Fama-French 3 factors (daily), U.S. and Europe

The files are zipped CSVs with text headers and footers; the helper keeps only the daily table.
Values are in **percent**.

In [12]:
FF_BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
FF_FILES = {
    "ff3_us_daily": "F-F_Research_Data_Factors_daily_CSV.zip",
    "ff3_europe_daily": "Europe_3_Factors_Daily_CSV.zip",
}

def get_french(zip_name):
    r = requests.get(FF_BASE + zip_name, timeout=60)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        text = z.read(z.namelist()[0]).decode("latin-1")
    lines = text.splitlines()
    # header = the first line that contains "Mkt-RF"; data = the 8-digit dates right after it
    start = next(i for i, l in enumerate(lines) if "Mkt-RF" in l)
    rows = []
    for l in lines[start + 1:]:
        parts = [p.strip() for p in l.split(",")]
        if not (parts[0].isdigit() and len(parts[0]) == 8):
            break
        rows.append(parts)
    cols = ["date"] + [c.strip() for c in lines[start].split(",")[1:]]
    df = pd.DataFrame(rows, columns=cols)
    df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")
    return df.set_index("date").astype(float)

for name, file in FF_FILES.items():
    ff = get_french(file).loc[START:END]
    ff.to_csv(RAW / f"{name}.csv")
    print(name, ff.shape, ff.index.min().date(), "->", ff.index.max().date())

ff3_us_daily (3897, 4) 2010-01-04 -> 2025-07-01
ff3_europe_daily (4043, 4) 2010-01-01 -> 2025-07-01


## 5. Download log

Websites change: record when the data was fetched and with which package versions.

In [14]:
log = {
    "downloaded_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "sample": [START, END],
    "pandas": pd.__version__,
    "yfinance": yf.__version__,
    "files": sorted(p.name for p in RAW.glob("*.csv")),
    "tickers_without_data": missing,
}
(RAW / "download_log.json").write_text(json.dumps(log, indent=2))
log

{'downloaded_utc': '2026-09-30T15:36:36+00:00',
 'sample': ['2010-01-01', '2025-07-01'],
 'pandas': '3.0.1',
 'yfinance': '1.7.0',
 'files': ['ff3_europe_daily.csv',
  'ff3_us_daily.csv',
  'fred_controls_daily.csv',
  'mccc_daily.csv',
  'prices_daily.csv',
  'universe.csv'],
 'tickers_without_data': []}